In [1]:
import anndata as ad
import decoupler as dc
import mofax as mfx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.stats.multitest import multipletests
import statsmodels.stats.multitest as multitest
from scipy.stats import mannwhitneyu

In [2]:
PATH_ADATA = "/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/adata/mofa_adata_30f.h5ad"
adata = ad.read_h5ad(PATH_ADATA)

factors = adata.uns["mofa_weights_factors"]
collectri = dc.get_collectri(organism="human")
tfs_of_interes = ["YAP1", "TEAD1", "TEAD2", "TEAD4", "WWTR1"]


# Calcular actividad de TFs YAP/TAZ por linea celular usando los pesos de MOFA
W = pd.DataFrame(
    adata.uns["mofa_weights"],
    index=adata.uns[f"mofa_weights_genes"],
    columns=factors
)
acts, pvals = dc.run_ulm(W.T, net=collectri, source="source", target="target", weight="weight")
acts_sig = acts.where((pvals < 0.05) & (acts < 0))
results = {
    "acts_sig": acts_sig[tfs_of_interes],
}
resultados = pd.DataFrame(results['acts_sig'])
resultados = resultados.dropna(how = 'all')

AQUI ES DONDE EMPIEZO A UTILIZAR LOS DATOS DE DEPMAP 

In [3]:
import requests
import os

prism_article_id = '25917643'
crispr_article_id = '25880521'
ruta_salida = "/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/"

for article_id, nombres in [
    (prism_article_id, ['Repurposing_Public_24Q2_LFC.csv',
                        'Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv',
                        'Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv',
                        'Repurposing_Public_24Q2_Treatment_Meta_Data.csv',
                        'Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv']),
    (crispr_article_id, ['CRISPRGeneDependency.csv'])
]:
    response = requests.get(f'https://api.figshare.com/v2/articles/{article_id}/files')
    files = response.json()
    for f in files:
        if f['name'] in nombres:
            print(f"Descargando {f['name']}...")
            r = requests.get(f['download_url'])
            with open(os.path.join(ruta_salida, f['name']), 'wb') as out:
                out.write(r.content)
            print(f"Guardado: {f['name']}")


Descargando Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv...
Guardado: Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv
Descargando Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv...
Guardado: Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv
Descargando Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv...
Guardado: Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv
Descargando Repurposing_Public_24Q2_LFC.csv...
Guardado: Repurposing_Public_24Q2_LFC.csv
Descargando CRISPRGeneDependency.csv...
Guardado: CRISPRGeneDependency.csv


In [4]:
cell_line_metadata = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv")
metadata = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv")
epdm = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv")
crispr = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/CRISPRGeneDependency.csv", index_col=0)
print(f"CRISPR: {crispr.shape}, DepMap: {epdm.shape}, metadata : {metadata.shape}" )

CRISPR: (1150, 18443), DepMap: (6790, 920), metadata : (6790, 7)


In [5]:
# Definir líneas celulares YAP/TAZ-dependientes usando CRISPR Gene  Dependency 
# Umbral > 0.5: líneas que necesitan ese gen para sobrevivir
yap_col = [col for col in crispr.columns if col.startswith("YAP1")][0]
wwtr1_col = [col for col in crispr.columns if col.startswith("WWTR1")][0]
tead_cols = [col for col in crispr.columns if any(col.startswith(t) for t in ["TEAD1", "TEAD2", "TEAD4"])]
# Pongo el limite en el 0.5 ya que me indica que silenciar ese gen reduce la supervivencia de la linea celular, lo que sugiere dependencia. Es un umbral comúnmente usado en análisis de dependencia genética.
yap_dependent = crispr[
    (crispr[yap_col] >  0.5) |
    (crispr[wwtr1_col] > 0.5) |
    crispr[tead_cols].gt(0.5).any(axis=1)
].index.tolist()

print(f"Líneas YAP/TAZ-dependientes: {len(yap_dependent)}")

Líneas YAP/TAZ-dependientes: 619


In [6]:
# Preparar tabla de DepMap en formato largo: depmap_id x BRD_ID → LFC
epm = epdm.rename(columns={'Unnamed: 0': 'BRD_ID'}).set_index('BRD_ID')
rep_clean = metadata[['IDs', 'Drug.Name', 'MOA', 'Synonyms']].drop_duplicates('IDs').rename(columns={'IDs': 'BRD_ID'})
epm_t = epm.T
epm_t.index.name = 'depmap_id' 
epm_t = epm_t.reset_index().merge(cell_line_metadata[['depmap_id', 'ccle_name']], on='depmap_id', how='left')
epm_t['cell_line'] = epm_t['ccle_name'].str.split('_').str[0] 


In [28]:
epm_final

,depmap_id,cell_line,BRD_ID,LFC,Drug.Name,MOA
0,ACH-000001,NIHOVCAR3,BRD:BRD-A00047421-001-01-7,-1.207281,ARV-825,BROMODOMAIN INHIBITOR
2,ACH-000002,HL60,BRD:BRD-A00047421-001-01-7,-4.231563,ARV-825,BROMODOMAIN INHIBITOR
4,ACH-000004,HEL,BRD:BRD-A00047421-001-01-7,-3.860672,ARV-825,BROMODOMAIN INHIBITOR
6,ACH-000005,HEL9217,BRD:BRD-A00047421-001-01-7,-2.271411,ARV-825,BROMODOMAIN INHIBITOR
8,ACH-000006,MONOMAC6,BRD:BRD-A00047421-001-01-7,0.277833,ARV-825,BROMODOMAIN INHIBITOR
...,...,...,...,...,...,...
8655954,ACH-001239,WM2664,BRD:BRD-U51753822-000-01-1,-0.273509,PHOSPHATIDYLCHOLINE,NaN
8655956,ACH-001306,8305C,BRD:BRD-U51753822-000-01-1,0.160925,PHOSPHATIDYLCHOLINE,NaN
8655958,ACH-001307,8505C,BRD:BRD-U51753822-000-01-1,-0.073471,PHOSPHATIDYLCHOLINE,NaN
8655960,ACH-001318,PLCPRF5,BRD:BRD-U51753822-000-01-1,-0.092772,PHOSPHATIDYLCHOLINE,NaN


In [7]:
# Pasar a formato largo excluyendo columnas de metadata
brd_cols = [c for c in epm_t.columns if c not in ['depmap_id', 'ccle_name', 'cell_line', 'index']]
epm_long = epm_t.melt(id_vars=['depmap_id', 'cell_line'], value_vars=brd_cols, var_name='BRD_ID', value_name='LFC').dropna(subset=['LFC'])
epm_final = epm_long.merge(rep_clean[['BRD_ID', 'Drug.Name','MOA']], on='BRD_ID').drop_duplicates()

In [19]:
prism_lines = epm_final['depmap_id'].unique()
n_yap = pd.Series(prism_lines).isin(yap_dependent).sum()
no_yap = (~pd.Series(prism_lines).isin(yap_dependent)).sum()
porcentaje = (n_yap/len(yap_dependent))*100
print(f"De todas las lineas celulares dependientes de yap que son {len(yap_dependent)}, de esas solo el {porcentaje}% pertenecen a prism.\nPor otro lado las lineas celulares que son yap dependientes {n_yap}\nMientras las que son no dependientes de yap son {no_yap}\nLas lineas celulares totales de PRISM son {len(prism_lines)}")



De todas las lineas celulares dependientes de yap que son 619, de esas solo el 65.75121163166398% pertenecen a prism.
Por otro lado las lineas celulares que son yap dependientes 407
Mientras las que son no dependientes de yap son 512
Las lineas celulares totales de PRISM son 919


In [8]:
# Análisis principal: ¿las drogas que inhiben YAP/TAZ según MOFA matan más
# las células YAP-dependientes en DepMap que las no dependientes?

# Scores de MOFA por droga (media entre concentraciones y placas)
scores = pd.DataFrame(adata.X, index=adata.obs_names, columns=adata.var_names)
scores['drug'] = adata.obs['drug']
scores_droga = scores.groupby('drug', observed=True).mean()

# Top 10 drogas más negativas por factor
top_drogas_por_factor = {
    factor: scores_droga[factor].nsmallest(10).index.tolist()
    for factor in resultados.index.unique()
}

resultados_yap = []
for factor, drogas in top_drogas_por_factor.items():
    brd_ids = rep_clean[rep_clean['Drug.Name'].isin(drogas) |
                        rep_clean['Synonyms'].apply(lambda x: any(d in str(x) for d in drogas))]['BRD_ID'].tolist() # Miro si mis drogas estan dentro de los  metadatos de las drogas de DepMap

    lfc_yap = epm_final[
        (epm_final['BRD_ID'].isin(brd_ids)) &
        (epm_final['depmap_id'].isin(yap_dependent))
    ]['LFC'].dropna()

    lfc_no_yap = epm_final[
        (epm_final['BRD_ID'].isin(brd_ids)) &
        (~epm_final['depmap_id'].isin(yap_dependent))
    ]['LFC'].dropna()

    if len(lfc_yap) < 20 or len(lfc_no_yap) < 20:
        continue

    stat, pval = mannwhitneyu(lfc_yap, lfc_no_yap, alternative='less')

    resultados_yap.append({
        'factor': factor,
        'lfc_medio_yap': lfc_yap.mean(),
        'lfc_medio_no_yap': lfc_no_yap.mean(),
        'pval': pval,
        'n_drogas': len(drogas)
    })

resultados_df = pd.DataFrame(resultados_yap)
mask = resultados_df['pval'].notna()
resultados_df.loc[mask, 'padj'] = multipletests(resultados_df.loc[mask, 'pval'], method='fdr_bh')[1]
print(resultados_df.sort_values('pval'))


     factor  lfc_medio_yap  lfc_medio_no_yap          pval  n_drogas  \
2   Factor6      -2.141610         -1.870308  3.029902e-08        10   
3   Factor8      -1.098752         -0.884937  3.936398e-04        10   
4  Factor25      -2.120097         -1.968452  3.035561e-03        10   
1   Factor5      -1.946165         -1.904468  2.605980e-01        10   
0   Factor3      -1.000383         -0.984109  7.091047e-01        10   
5  Factor27      -1.745251         -1.985629  9.773973e-01        10   

           padj  
2  1.817941e-07  
3  1.180919e-03  
4  6.071122e-03  
1  3.908970e-01  
0  8.509256e-01  
5  9.773973e-01  


In [10]:
# Factores significativos
factores_sig = ['Factor6','Factor8','Factor25']
factores_sig = resultados_df[resultados_df['padj'] < 0.05]['factor'].tolist()

# Drogas candidatas
candidatas = {}
for factor in factores_sig:
    candidatas[factor] = top_drogas_por_factor[factor]

print(candidatas)

{'Factor6': ['DINACICLIB', 'HOMOHARRINGTONINE', 'SBI-0640756', 'HARRINGTONINE', 'BELZUTIFAN', 'TAK-901', 'PEMIGATINIB', 'OUABAIN_(OCTAHYDRATE)', 'DIGITOXIN', 'HYDROXYFASUDIL'], 'Factor8': ['ELIMUSERTIB_HYDROCHLORIDE', 'TRAMETINIB_(DMSO_TF_SOLVATE)', 'TRAMETINIB', 'BINIMETINIB', 'BI-3406', 'COBIMETINIB', 'RMC-6236', 'BIMIRALISIB', 'PONATINIB', 'NG25'], 'Factor25': ['DAPTOMYCIN', 'VINCRISTINE', 'PERETINOIN', 'VINBLASTINE_(SULFATE)', 'PONATINIB', 'HYDROXYFASUDIL', 'TOFACITINIB', 'AURANOFIN', 'IXAZOMIB', 'IPATASERTIB']}


In [ ]:
rows = []
factores_sig = ['Factor6', 'Factor8', 'Factor25']

for factor in factores_sig:
    drogas = top_drogas_por_factor[factor]
    for drug in drogas:
        brd = rep_clean[
            (rep_clean['Drug.Name'] == drug) |
            rep_clean['Synonyms'].apply(lambda x: drug in str(x))
        ]['BRD_ID'].tolist()
        
        lfc_yap = epm_final[
            epm_final['BRD_ID'].isin(brd) &
            epm_final['depmap_id'].isin(yap_dependent)
        ]['LFC'].mean()
        
        lfc_no_yap = epm_final[
            epm_final['BRD_ID'].isin(brd) &
            ~epm_final['depmap_id'].isin(yap_dependent)
        ]['LFC'].mean()
        
        rows.append({
            'factor': factor,
            'drug': drug,
            'lfc_yap': lfc_yap,
            'lfc_no_yap': lfc_no_yap,
            'diff': lfc_yap - lfc_no_yap
        })

drug_df = pd.DataFrame(rows).sort_values(['factor', 'diff'])

# MOA desde rep_clean, sin duplicados
moa_info = rep_clean[['Drug.Name', 'MOA']].drop_duplicates('Drug.Name')

# Merge y eliminar columna Drug.Name duplicada
resultado = drug_df.merge(
    moa_info,
    left_on='drug',
    right_on='Drug.Name',
    how='left'
).drop(columns='Drug.Name').dropna()  # ← elimina la columna duplicada

resultado.to_excel("/home/ulc/co/mao/resultados_yap")

      factor               drug   lfc_yap  lfc_no_yap      diff  \
0   Factor25        VINCRISTINE -2.026184   -1.766984 -0.259199   
1   Factor25           IXAZOMIB -4.302761   -4.132076 -0.170684   
2   Factor25          PONATINIB -2.166900   -2.037726 -0.129174   
3   Factor25          AURANOFIN -5.750349   -5.634087 -0.116262   
4   Factor25     HYDROXYFASUDIL -0.474437   -0.363313 -0.111124   
5   Factor25         PERETINOIN -0.242269   -0.197174 -0.045095   
6   Factor25        TOFACITINIB -0.059909   -0.018430 -0.041480   
7   Factor25         DAPTOMYCIN -0.022120    0.008165 -0.030285   
10   Factor6          DIGITOXIN -3.356850   -3.153068 -0.203782   
11   Factor6         DINACICLIB -4.074832   -3.896853 -0.177979   
12   Factor6            TAK-901 -2.037138   -1.880810 -0.156328   
14   Factor6     HYDROXYFASUDIL -0.474437   -0.363313 -0.111124   
15   Factor6  HOMOHARRINGTONINE -4.181257   -4.085220 -0.096036   
16   Factor6      HARRINGTONINE -3.716764   -3.634870 -0.08189

In [18]:

resultado.to_excel("/home/ulc/co/mao/resultados_yap.xlsx")